# Source physics: power, derived quantities, throughput and honest gaps

A lithography source is judged by much more than its wavelength: average power
at a defined plane, bandwidth, coherence, pulse-to-pulse stability and, above
all, whether it exists. This notebook:

1. maps every preset on a power-vs-wavelength landscape;
2. reads the **derived quantities**, physics computed from machine parameters;
3. turns power into dose-limited **wafer throughput** (🔶);
4. explores three parameter sensitivities (FEL energy spread, SSMB bunching,
   HHG driver power);
5. tabulates the **gaps** between the theoretical (🧪) sources and production;
6. relates wavelength to photon shot noise.

All numbers come live from the source models. Machine parameters in the presets
are illustrative and anchored to published figures where the
[source pages](../../docs/sources/index.md) say so; they are not the
specifications of any facility.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import highuvlith as huv
from highuvlith import viz

%matplotlib inline

## 1. The landscape

`viz.plot_source_landscape()` plots log power vs log wavelength for the presets of
`huv.source_landscape()`, with the 250 W to 1 kW EUV HVM band (in-band power at
the intermediate focus) for reference. Marker style encodes maturity
(demonstrated, projection, theoretical); sources without a modeled power (Hg
lamps, the entangled-photon source) are listed but not drawn.

Read the axis carefully: powers are each family's own definition (IF for
plasmas, output for lasers and FELs, filtered 4π X-ray power for tubes), so
points are comparable only within the caveats of the `power_definition`
column.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
viz.plot_source_landscape(ax=ax);

## 2. Derived quantities

Every source reports `derived_quantities()`: a list of `(name, value, unit,
note)` computed from its machine parameters. The note states the formula or the
assumption. Two examples: the tin LPP power chain (drive laser → conversion
efficiency → collection → étendue → IF) and the FLASH-like SASE FEL (Pierce
parameter, gain length, saturation).

In [ ]:
def show_derived(src, names=None):
    for name, value, unit, note in src.derived_quantities():
        if names is None or name in names:
            print(f"  {name:32s} {value:11.4g} {unit:10s} {note[:70]}")

print("Sn LPP (lpp_sn_13nm5):")
show_derived(huv.SourceConfig.lpp_sn_13nm5())
print("\nFLASH-like XFEL (xfel_flash_13nm5):")
show_derived(huv.SourceConfig.xfel_flash_13nm5(),
             {"pierce_parameter_1d", "gain_length_1d", "gain_length_3d", "ming_xie_lambda",
              "saturation_length_3d", "sase_bandwidth_from_rho", "longitudinal_modes", "average_power"})

## 3. Dose-limited wafer throughput (🔶)

`huv.wafer_throughput` converts a source's average power into wafers per hour
with a deliberately simple scanner model: one optics-train transmission (by
default 10 Mo/Si mirrors at R 0.70 and a reflective mask at 0.65 for EUV; 0.30
and 0.90 for refractive DUV/VUV), a field layout, lumped overheads and an
optional stage limit. The status and notes travel with the result.

Only projection sources are meaningful here. With the default `preset="auto"`
the function raises `ValueError` when no projection-lithography optics train
covers the wavelength (X-ray tubes and the betatron are LIGA/proximity sources;
the 25–47 nm LPA-FEL, Ar HHG and Ar soft-X-ray-laser lines lie outside the
Mo/Si and La/B mirror bands), and returns `None` for the Hg lamps and the
entangled-photon source, which have no modeled average power.

In [ ]:
rows, no_optics = [], []
for p in huv.source_landscape():
    if p.average_power_w is None:
        continue
    try:
        t = huv.wafer_throughput(eval("huv.SourceConfig." + p.factory), dose_mj_cm2=30.0)
    except ValueError:  # no projection optics preset at this wavelength
        no_optics.append(p.label)
        continue
    rows.append((p.label, p.wavelength_nm, p.average_power_w, t["preset"], t["power_at_wafer_w"],
                 t["wafers_per_hour"], p.maturity))
print(f"{'preset':22s} {'lambda/nm':>9s} {'P_src/W':>10s} {'train':>10s} {'P_wafer/W':>10s} {'WPH':>9s}  maturity")
for label, wl, p_src, preset, p_w, wph, maturity in sorted(rows, key=lambda r: -r[5]):
    print(f"{label:22s} {wl:9.4g} {p_src:10.3g} {preset:>10s} {p_w:10.3g} {wph:9.3g}  {maturity}")
print("\nNo projection optics preset:", ", ".join(no_optics))
lpp = huv.wafer_throughput(huv.SourceConfig.lpp_sn_13nm5(), dose_mj_cm2=30.0)
print("\nSn LPP notes:", lpp["status"], "-", "; ".join(lpp["notes"]))

The 250 W tin LPP gives about 150 wafers per hour at 30 mJ/cm² with these
illustrative scanner numbers, the right order for an NXE:3400-class tool.
Doubling the source power does not double throughput, because the per-field and
per-wafer overheads remain. Sources marked "projection" reach high numbers only
on paper.

Dose and throughput trade directly: a less sensitive (higher-dose) resist,
often chosen to reduce stochastic noise, costs wafers per hour.

In [ ]:
doses = np.linspace(10.0, 90.0, 17)
fig, ax = plt.subplots(figsize=(7, 4))
for factory, label in (("lpp_sn_13nm5", "Sn LPP 250 W"), ("lpp_sn_13nm5_500w", "Sn LPP 500 W"),
                       ("dpp_sn_13nm5", "Sn DPP")):
    src = getattr(huv.SourceConfig, factory)()
    ax.plot(doses, [huv.wafer_throughput(src, dose_mj_cm2=float(d))["wafers_per_hour"] for d in doses],
            label=label)
ax.set(xlabel="resist dose (mJ/cm²)", ylabel="wafers per hour", title="Dose-limited throughput (EUV train)")
ax.legend()
ax.grid(alpha=0.3)

## 4. Parameter sensitivities

### FEL gain and the energy-spread criterion

A free-electron laser amplifies only if the electron beam's relative energy
spread $\sigma_\delta$ stays below the Pierce parameter ρ. The Ming Xie fit
quantifies the 3D degradation; the saturation length grows rapidly once
$\sigma_\delta / \rho$ approaches 1. Laser-plasma accelerators naturally produce
percent-level spreads, which is why the 25 nm LPA-FEL preset is a design
projection: its illustrative 1 % spread violates the criterion and the 4 m
undulator is far shorter than the derived saturation length.

In [ ]:
spreads = np.array([1e-3, 2e-3, 3e-3, 5e-3, 7e-3, 1e-2, 2e-2])
ratio, lsat = [], []
for es in spreads:
    fel = huv.SourceConfig.lpa_fel(25.0, undulator_period_mm=20.0, undulator_k=1.6724,
                                   energy_spread_rel=float(es))
    ratio.append(fel.derived_quantity("energy_spread_over_rho"))
    lsat.append(fel.derived_quantity("saturation_length_3d"))
fig, ax = plt.subplots(figsize=(7, 4))
ax.loglog(spreads * 100, lsat, "o-")
ax.axhline(4.0, ls="--", color="grey")
ax.text(0.4, 4.4, "undulator length 4 m", color="grey", fontsize=9)
for x, y, r in zip(spreads * 100, lsat, ratio):
    ax.annotate(f"σδ/ρ = {r:.2f}", (x, y), textcoords="offset points", xytext=(6, -10), fontsize=8)
ax.set(xlabel="relative energy spread (%)", ylabel="3D saturation length (m)",
       title="25 nm LPA-FEL: gain vs electron energy spread")
ax.grid(alpha=0.3, which="both")
bella = huv.SourceConfig.lpa_fel_bella_25nm()
print(f"preset: bandwidth {bella.bandwidth_pm:.1f} pm (= 2 rho lambda), "
      f"sigma_delta/rho = {bella.derived_quantity('energy_spread_over_rho'):.2f}")

### SSMB: power scales with the bunching factor squared

Steady-state microbunching would make a storage ring radiate coherently at a high
harmonic of a modulation laser. The coherent power scales as $\lvert b \rvert^2$, where
$b$ is the bunching factor at the target harmonic, which requires microbunches a
few nanometres long on **every** turn. The 1 kW preset needs $b \approx 0.15$, never
demonstrated at EUV.

In [ ]:
bs = np.linspace(0.01, 0.25, 25)
power = [huv.SourceConfig.ssmb(bunching_factor=float(b)).average_power_w for b in bs]
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(bs, power)
ax.axhline(250.0, ls="--", color="grey")
ax.text(0.012, 280, "250 W HVM reference", color="grey", fontsize=9)
ax.set(xlabel="bunching factor b", ylabel="coherent power (W)", yscale="log",
       title="SSMB 13.5 nm: P proportional to b²")
ax.grid(alpha=0.3, which="both")
ssmb = huv.SourceConfig.ssmb_euv_13nm5()
print(f"b for 250 W: {ssmb.derived_quantity('bunching_for_hvm'):.3f}; "
      f"microbunch length at the preset b: {ssmb.derived_quantity('microbunch_rms_length'):.1f} nm")

### HHG: what a 250 W harmonic would need

High-harmonic power is a driver power times a small conversion efficiency that
falls steeply with harmonic order. The model's efficiency is an
order-of-magnitude estimate (±1 decade), and each preset is compared against
a curve of measured records at its photon energy.

In [ ]:
for label, src in (("Ne q59 13.56 nm (5 W driver)", huv.SourceConfig.hhg()),
                   ("Ne q59 preset", huv.SourceConfig.hhg_ne_13nm5()),
                   ("Ar 30 nm preset", huv.SourceConfig.hhg_ar_30nm())):
    print(f"{label:30s} P = {src.average_power_w:.2g} W, efficiency {src.derived_quantity('conversion_efficiency'):.1e}, "
          f"driver for 250 W: {src.derived_quantity('driver_power_for_hvm'):.1e} W, "
          f"vs measured record x{src.derived_quantity('power_vs_measured_record'):.2f}")

## 5. The honest gaps

The 🧪 families are runnable research projections. Their own derived quantities
say how far each is from a production EUV source (250 W at IF) or from a usable
exposure.

In [ ]:
gaps = [
    ("Inverse Compton (ics_compact_euv_13nm5)", huv.SourceConfig.ics_compact_euv_13nm5(), "hvm_power_gap", "x short of 250 W"),
    ("Smith-Purcell (smith_purcell)", huv.SourceConfig.smith_purcell(), "hvm_power_gap", "x short of 250 W"),
    ("Betatron (betatron)", huv.SourceConfig.betatron(), "hvm_power_gap", "x short of 250 W (context only)"),
    ("SSMB (ssmb_euv_13nm5)", huv.SourceConfig.ssmb_euv_13nm5(), "bunching_factor", "bunching factor assumed every turn"),
    ("Gd LPP 6.7 nm (lpp_gd_6nm7)", huv.SourceConfig.lpp_gd_6nm7(), "hvm_power_ratio", "of 250 W (a projection itself)"),
    ("CW-SC XFEL (xfel_cw_sc_13nm5)", huv.SourceConfig.xfel_cw_sc_13nm5(), "hvm_power_ratio", "of 250 W (design projection)"),
    ("ERL FEL (xfel_erl_13nm5)", huv.SourceConfig.xfel_erl_13nm5(), "beam_power_average", "W of beam to energy-recover"),
    ("Entangled N00N (entangled_noon)", huv.SourceConfig.entangled_noon(), "exposure_time_bound", "s to clear 1 cm² (lower bound)"),
]
for label, src, key, meaning in gaps:
    print(f"{label:42s} {key:22s} {src.derived_quantity(key):10.3g}  {meaning}")

The entangled-photon number deserves a sentence: even at the loosest
independent bound on the entangled two-photon absorption cross-section, clearing
one square centimetre at the preset pair rate takes about $10^{16}$ s. The
quantum-imaging models themselves (N-photon absorption vs the ideal N00N limit)
are covered in the [Python API reference](../../docs/python-api.md) and the
[research modules](../../docs/research-modules.md) page; both are 🧪.

## 6. Wavelength and photon shot noise

At the same dose, shorter wavelengths deliver fewer photons, because each photon
carries $hc / \lambda$. The number of photons landing on a small square sets a
Poisson noise floor $1 / \sqrt{N}$ on the local dose, one root of line-edge
roughness at EUV.

In [ ]:
side = 16.0
print(f"incident photons in a {side:.0f} nm square at 30 mJ/cm²:")
for label, wl in (("ArF", 193.368), ("F2", 157.63), ("EUV", 13.5), ("BEUV", 6.7)):
    n = huv.photons_per_square(30.0, wl, side)
    print(f"  {label:5s} {wl:8.3f} nm: {n:8.0f} photons, relative noise {huv.relative_shot_noise(30.0, wl, side) * 100:.2f} %")

Docs: [Light sources](../../docs/sources/index.md),
[capability matrix](../../docs/capability-matrix.md) (Light sources section),
[Python API: sources](../../docs/python-api.md#sources-landscape-and-throughput).